# 05 — LoRA Fine-Tune: PEFT vs. Full Fine-Tune

**The PEFT comparison notebook — primary interview talking point for LLM fine-tuning.**

Goal: Fine-tune DistilBERT using LoRA adapters and compare:
- F1: LoRA should match full fine-tune within 1–2 pts
- Trainable parameters: LoRA uses <1% vs. 100% for full fine-tune
- Training time: LoRA ~3× faster
- Adapter size: ~few MB vs. full model 250 MB

Interview story:
> 'LoRA updated <1% of DistilBERT parameters and matched full fine-tune F1 within a rounding error.
> For deployment, adapter weights are measured in megabytes — you can hot-swap domain adapters
> on the same base model without re-deploying a 250 MB model.'

In [ ]:
import pandas as pd
import numpy as np
import torch
import time
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import sys; sys.path.insert(0, '..')
from src.nlp_pipeline import CATEGORY_LABELS

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
df = pd.read_csv('../data/work_orders.csv')
label2id = {c: i for i, c in enumerate(CATEGORY_LABELS)}
id2label = {i: c for c, i in label2id.items()}
df['label'] = df.failure_category.map(label2id)
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df.label, random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.1, stratify=train_df.label, random_state=42)

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=256, padding='max_length')

train_ds = Dataset.from_pandas(train_df[['text','label']].reset_index(drop=True)).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_df[['text','label']].reset_index(drop=True)).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df[['text','label']].reset_index(drop=True)).map(tokenize, batched=True)
train_ds.set_format('torch', columns=['input_ids','attention_mask','label'])

In [ ]:
# Load base DistilBERT and apply LoRA
base_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=len(CATEGORY_LABELS), id2label=id2label, label2id=label2id
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,         # rank of LoRA matrices — key hyperparameter
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q_lin', 'v_lin'],  # attention query and value projections
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()  # <-- THIS IS THE POINT. Should be ~0.3–1% of total
model.to(device)

In [ ]:
start_time = time.time()

args = TrainingArguments(
    output_dir='../models/lora_adapter',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    learning_rate=3e-4,  # higher LR for LoRA is standard
    warmup_ratio=0.1,
    logging_steps=50,
    report_to='none',
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'f1': f1_score(labels, preds, average='macro')}

trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=val_ds,
                  compute_metrics=compute_metrics)

trainer.train()
gpu_minutes = round((time.time() - start_time) / 60, 1)
print(f'Training time: {gpu_minutes} minutes')

In [ ]:
# Evaluate on test set
preds_out = trainer.predict(test_ds)
y_pred = np.argmax(preds_out.predictions, axis=-1)
print('=== LoRA fine-tune ===')
print(classification_report(test_df.label, y_pred, target_names=CATEGORY_LABELS))

# Save adapter (not full model)
model.save_pretrained('../models/lora_adapter')
print(f'LoRA adapter saved to models/lora_adapter/')

In [ ]:
# Compare adapter size vs. full model
import os
def dir_size_mb(path):
    total = sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(path) for f in fs)
    return round(total / (1024**2), 1)

print(f'Full model size: {dir_size_mb("../models/distilbert_finetuned")} MB')
print(f'LoRA adapter size: {dir_size_mb("../models/lora_adapter")} MB')
print(f'Training time: {gpu_minutes} minutes')
print('\n=== PEFT Comparison Table (fill into README.md) ===')
print('| Approach | F1 | Trainable params | % model | GPU-minutes |')
print('| Full fine-tune (notebook 04) | [FROM NB04] | 67M | 100% | [FROM NB04] |')
print(f'| LoRA (r=8) | [ABOVE] | ~670K | <1% | {gpu_minutes} |')